In [ ]:
# 1. Import Required Libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.callbacks import Callback, LearningRateScheduler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os
from collections import Counter

# 2. Define Dataset Path & Parameters
data_dir = "E:/Projects/Strabismus/Denoised"

img_height, img_width = 224, 224
batch_size = 32
num_classes = 5  # Since we have 5 categories

# 3. Data Preprocessing & Augmentation
datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.15,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# 4. Compute Class Weights
class_weights = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_generator.classes)
class_weight_dict = dict(enumerate(class_weights))

# 5. Load Pretrained VGG19 Model
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
base_model.trainable = False  # Freeze base model layers

# 6. Define Modified VGG19 Model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(1024, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# 7. Compile the Model
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# 8. Define Learning Rate Scheduler
def lr_scheduler(epoch, lr):
    return lr * 0.95 if epoch > 5 else lr
callback_lr = LearningRateScheduler(lr_scheduler)

# 9. Define Optimized Metrics Callback
class MetricsCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = model.predict(val_generator, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = val_generator.classes[:len(y_pred)]
        
        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=1)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=1)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=1)
        
        print(f'\nEpoch {epoch+1} Metrics:')
        print(f'Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')

# 10. Train the Model
model.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[MetricsCallback(), callback_lr]
)

# 11. Fine-Tune Last Few Layers
base_model.trainable = True
for layer in base_model.layers[:-5]:  # Unfreeze last few layers
    layer.trainable = False

# Recompile with lower learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

# 12. Continue Training with Fine-Tuning
model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[MetricsCallback(), callback_lr]
)


Found 4395 images belonging to 5 classes.
Found 775 images belonging to 5 classes.
Epoch 1/50
138/138 [==============================] - ETA: 0s - loss: 2.3312 - accuracy: 0.2266
Epoch 1 Metrics:
Accuracy: 0.1935, Precision: 0.1523, Recall: 0.1904, F1 Score: 0.1586
138/138 [==============================] - 81s 582ms/step - loss: 2.3312 - accuracy: 0.2266 - val_loss: 2.2544 - val_accuracy: 0.3277 - lr: 0.0010
Epoch 2/50
138/138 [==============================] - ETA: 0s - loss: 2.2890 - accuracy: 0.2428
Epoch 2 Metrics:
Accuracy: 0.1961, Precision: 0.6796, Recall: 0.1970, F1 Score: 0.1081
138/138 [==============================] - 44s 319ms/step - loss: 2.2890 - accuracy: 0.2428 - val_loss: 2.2477 - val_accuracy: 0.3161 - lr: 0.0010
Epoch 3/50
138/138 [==============================] - ETA: 0s - loss: 2.2648 - accuracy: 0.2567
Epoch 3 Metrics:
Accuracy: 0.1910, Precision: 0.1876, Recall: 0.1874, F1 Score: 0.1416
138/138 [==============================] - 42s 307ms/step - loss: 2.2648 -